# WeedDet v4 — Training Notebook
**AgriNav | Benny Merryman-Smith**

Loads `weeddet_for_VSCode.py` from Google Drive — no GitHub required.

| Cell | Purpose |
|------|---------|
| 0 | Mount Drive + load script from Drive |
| 1 | Extract dataset + pre-training inference check |
| 2 | Score diagnostic (confirms old ckpt vs new preprocessing) |
| 3 | **Train WeedDet v4** |
| 4 | Post-training visual check |


## Cell 0 — Mount Drive + Load Script
> ✏️ Set `SCRIPT_DIR` to the Drive folder containing your WeedDet script.  
> The script must be named `weeddet_Latest.py` for the current imports to work correctly.

In [ ]:
from google.colab import drive
import sys, os

drive.mount('/content/drive')
SCRIPT_DIR = '/content/drive/MyDrive/weeddet_v2_checkpoints'

# Verify the file exists before adding to path
script_file = os.path.join(SCRIPT_DIR, 'weeddet_Latest.py')
assert os.path.exists(script_file), (
    f'Script not found: {script_file}\n'
    'Make sure weeddet_Latest.py is saved in that Drive folder.'
)

# sys.path takes directories, not file paths
sys.path.insert(0, SCRIPT_DIR)
print(f'Done — {script_file} on path')


## Cell 1 — Extract Dataset & Prepare Image for Post-Training Check
Extracts the dataset and prepares a random image for later visual checking after training finishes.

In [ ]:
import torch, random, glob, zipfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torchvision.transforms as T
from PIL import Image
import sys

# Clear stale imports
for _m in list(sys.modules.keys()):
    if 'weeddet' in _m.lower(): del sys.modules[_m]
# Correcting import name to match the script file: weeddet_Latest.py
import weeddet_Latest as wd

# ── Paths ─────────────────────────────────────────────────────────────────────
EXTRACT_DIR = '/content/dataset'
ZIP_PATH    = '/content/drive/MyDrive/weeddet_v2_checkpoints/rice_detection_for_export.v1i.voc.zip'

SCORE_THR = 0.50
NMS_THR   = 0.25

# ── Unzip dataset ─────────────────────────────────────────────────────────────
if not os.path.exists(EXTRACT_DIR) or len(os.listdir(EXTRACT_DIR)) == 0:
    print('rice_detection_for_export.v1i.voc.zip...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print('Done.')
else:
    print('Dataset already extracted.')

# ── Setup for image processing ────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Pick a random image and prepare for inference
all_imgs = (glob.glob(EXTRACT_DIR + '/**/*.jpg', recursive=True) +
            glob.glob(EXTRACT_DIR + '/**/*.png', recursive=True))
all_imgs = [p for p in all_imgs if 'checkpoint' not in p.lower()]
assert all_imgs, f'No images found in {EXTRACT_DIR}'

img_path = random.choice(all_imgs)
print(f'Selected: {img_path}')

img_orig           = Image.open(img_path).convert('RGB')
img_lb, scale, pl, pt = wd.letterbox_pil(img_orig, 512)
tf     = T.Compose([T.ToTensor(),
                    T.Normalize(mean=wd.IMAGENET_MEAN, std=wd.IMAGENET_STD)])
tensor = tf(img_lb).unsqueeze(0).to(device)

print('Old checkpoint baseline visual and comparison skipped as requested.')

## Cell 2 — Score Diagnostic Skipped
Score diagnostic for the old checkpoint is skipped as comparison is not desired.

## Cell 3 — Train WeedDet v4
Checkpoints saved as `weeddet_v4_best.pth` and `weeddet_v4_epoch{N}.pth`.

Expected output per epoch:
```
Epoch 01/60  avg_loss=0.87xx  best=inf  lr=0.001000
  ★ Best checkpoint -> .../weeddet_v4_best.pth
```

In [ ]:
import os, sys, glob, torch.nn as nn
from pathlib import Path

# ── Rename labels/ → annotations/ if needed ──────────────────────────────────
for split in ['train', 'valid', 'test', 'val']:
    src = f'/content/dataset/{split}/labels'
    dst = f'/content/dataset/{split}/annotations'
    if os.path.exists(src) and not os.path.exists(dst):
        os.rename(src, dst)
        print(f'Renamed {split}/labels → annotations')

# ── Build flat directory WeedDataset can actually read ────────────────────────
FLAT_ROOT = '/content/rice_flat'
os.makedirs(f'{FLAT_ROOT}/images',      exist_ok=True)
os.makedirs(f'{FLAT_ROOT}/annotations', exist_ok=True)

train_stems, val_stems = [], []

for split, stems_list in [('train', train_stems), ('valid', val_stems), ('val', val_stems)]:
    base_dir = Path(f'/content/dataset/{split}')
    if not base_dir.exists():
        continue

    # Check if images are in 'images/' or directly in the split folder
    if (base_dir / 'images').exists():
        img_dir = base_dir / 'images'
    else:
        img_dir = base_dir

    # Check if annotations are in 'annotations/' or directly in the split folder
    if (base_dir / 'annotations').exists():
        ann_dir = base_dir / 'annotations'
    else:
        ann_dir = base_dir

    if not img_dir.exists(): print(f'WARNING: {img_dir} missing'); continue
    if not ann_dir.exists(): print(f'WARNING: {ann_dir} missing'); continue

    for img_p in sorted(img_dir.glob('*')):
        if img_p.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
            continue
        xml_p = ann_dir / (img_p.stem + '.xml')
        if not xml_p.exists():
            continue
        dst_i = Path(FLAT_ROOT) / 'images'      / img_p.name
        dst_x = Path(FLAT_ROOT) / 'annotations' / (img_p.stem + '.xml')
        if not dst_i.exists(): os.symlink(img_p.resolve(), dst_i)
        if not dst_x.exists(): os.symlink(xml_p.resolve(), dst_x)
        stems_list.append(img_p.stem)

Path(f'{FLAT_ROOT}/train.txt').write_text('\n'.join(train_stems))
Path(f'{FLAT_ROOT}/val.txt'  ).write_text('\n'.join(val_stems))
print(f'Train : {len(train_stems)} pairs  |  Val : {len(val_stems)} pairs')
assert train_stems, 'No training images found — check /content/dataset structure'

# ── Sanity check — catches the "instant training" bug before wasting time ─────
for _m in list(sys.modules.keys()):
    if 'weeddet' in _m.lower(): del sys.modules[_m]
import weeddet_Latest as wd

probe  = wd.WeedDataset(FLAT_ROOT, 'train', img_size=512, augment=False)
sample = next((probe[i] for i in range(min(20, len(probe))) if probe[i] is not None), None)
assert sample is not None, \
    'All samples returned None — images not found at FLAT_ROOT/images/. Check symlinks.'
img_t, tgt = sample
assert img_t.shape == (3, 512, 512), f'Wrong shape {img_t.shape} — letterbox not applied'
print(f'Dataset check: {len(probe)} train samples  |  shape {img_t.shape}  |  '
      f'{len(tgt["boxes"])} GT boxes  ✓')

# ── Train ─────────────────────────────────────────────────────────────────────
CKPT_DIR_V4 = '/content/drive/MyDrive/weeddet_v4_checkpoints'
os.makedirs(CKPT_DIR_V4, exist_ok=True)

config = {
    'data_root'         : FLAT_ROOT,       # flat dir — NOT /content/dataset
    'checkpoint_dir'    : CKPT_DIR_V4,
    'checkpoint_prefix' : 'weeddet_v4',
    'num_classes'       : 1,
    'anchor_base_scale' : 6,
    'lsc_k'             : 7,
    'img_size'          : 512,
    'batch_size'        : 2,
    'base_lr'           : 0.001,
    'min_lr'            : 0.00005,
    'momentum'          : 0.9,
    'weight_decay'      : 0.0001,
    'scheduler'         : 'cosine',
    'num_epochs'        : 60,
    'warmup_iters'      : 300,
    'warmup_factor'     : 0.001,
    'freeze_bn'         : True,
    'grad_clip'         : 0.5,
    'augment'           : True,
    'save_every'        : 5,
    'num_workers'       : 2,
}

model_v4 = wd.WeedDet(
    num_classes       = config['num_classes'],
    anchor_base_scale = config['anchor_base_scale'],
    lsc_k             = config['lsc_k'],
).to(device)

if config['freeze_bn']:
    for m in model_v4.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()
            for p in m.parameters(): p.requires_grad = False
    print('BatchNorm frozen.')

total_p = sum(p.numel() for p in model_v4.parameters()) / 1e6
train_p = sum(p.numel() for p in model_v4.parameters() if p.requires_grad) / 1e6
print(f'Parameters : {total_p:.2f}M total  |  {train_p:.2f}M trainable')
print(f'Saving to  : {CKPT_DIR_V4}')
print()

trained_v4 = wd.train_with_progress(config)

print('\n' + '='*60)
print('WeedDet v4 training complete.')
print(f'Best checkpoint: {CKPT_DIR_V4}/weeddet_v4_best.pth')
print('='*60)

## Cell 4 — Post-Training Visual Check
Loads `weeddet_v4_best.pth` and runs inference on the same image used in Cell 1.  
**Green** = ground truth (if available) | **Red** = v4 predictions

In [ ]:
V4_CKPT   = f'{CKPT_DIR_V4}/weeddet_v4_best.pth'
SCORE_THR = 0.50
NMS_THR   = 0.25

if not os.path.exists(V4_CKPT):
    print(f'v4 checkpoint not found: {V4_CKPT}')
    print('Run Cell 3 first.')
else:
    for _m in list(sys.modules.keys()):
        if 'weeddet' in _m.lower(): del sys.modules[_m]
    import weeddet_for_VSCode as wd

    ckpt_v4  = torch.load(V4_CKPT, map_location=device)
    state_v4 = ckpt_v4.get('state_dict', ckpt_v4)
    model_v4 = wd.WeedDet(num_classes=1, anchor_base_scale=6, lsc_k=7).to(device)
    model_v4.load_state_dict(state_v4, strict=False)
    model_v4.eval()
    print(f'WeedDet v4 loaded — epoch {ckpt_v4.get("epoch","?")}  '
          f'loss={ckpt_v4.get("loss",float("nan")):.4f}')

    # img_path and tensor already set from Cell 1 — reuses same image
    with torch.no_grad():
        cls_l, regs, anchors, ishape = model_v4._get_logits(tensor)
        results = model_v4._decode(cls_l, regs, anchors, ishape,
                                   score_thr=0.05, nms_thr=NMS_THR,
                                   max_dets=300, output_thr=SCORE_THR)

    boxes_v4  = wd.unpad_boxes(results[0]['boxes'].cpu(), scale, pl, pt)
    scores_v4 = results[0]['scores'].cpu()

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(np.array(img_orig))
    for i, box in enumerate(boxes_v4):
        x1,y1,x2,y2 = box.tolist()
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                     linewidth=2, edgecolor='red', facecolor='none'))
        ax.text(x1, max(y1-4,0), f'rice {scores_v4[i]:.2f}',
                color='red', fontsize=8, backgroundcolor='black')
    ax.axis('off')
    ax.set_title(f'WeedDet v4 | {len(boxes_v4)} detections '
                 f'(score>={SCORE_THR}) | {img_path.split("/")[-1]}')
    plt.tight_layout(); plt.show()

    print(f'Detections : {len(boxes_v4)}')
    print(f'Score range: {scores_v4.min():.3f}-{scores_v4.max():.3f}'
          if len(scores_v4) else 'No detections above threshold')
    print(f'\nCheckpoints saved at: {CKPT_DIR_V4}')


In [ ]:
CKPT_DIR_V4 = '/content/drive/MyDrive/weeddet_v4_checkpoints'
V4_CKPT   = f'{CKPT_DIR_V4}/weeddet_v4_best.pth'
SCORE_THR = 0.50
NMS_THR   = 0.25

if not os.path.exists(V4_CKPT):
    print(f'v4 checkpoint not found: {V4_CKPT}')
    print('Run Cell 3 first.')
else:
    for _m in list(sys.modules.keys()):
        if 'weeddet' in _m.lower(): del sys.modules[_m]
    import weeddet_for_VSCode as wd

    ckpt_v4  = torch.load(V4_CKPT, map_location=device)
    state_v4 = ckpt_v4.get('state_dict', ckpt_v4)
    model_v4 = wd.WeedDet(num_classes=1, anchor_base_scale=6, lsc_k=7).to(device)
    model_v4.load_state_dict(state_v4, strict=False)
    model_v4.eval()
    print(f'WeedDet v4 loaded — epoch {ckpt_v4.get("epoch","?")}  '
          f'loss={ckpt_v4.get("loss",float("nan")):.4f}')

    # img_path and tensor already set from Cell 1 — reuses same image
    with torch.no_grad():
        cls_l, regs, anchors, ishape = model_v4._get_logits(tensor)
        results = model_v4._decode(cls_l, regs, anchors, ishape,
                                   score_thr=0.05, nms_thr=NMS_THR,
                                   max_dets=300, output_thr=SCORE_THR)

    boxes_v4  = wd.unpad_boxes(results[0]['boxes'].cpu(), scale, pl, pt)
    scores_v4 = results[0]['scores'].cpu()

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(np.array(img_orig))
    for i, box in enumerate(boxes_v4):
        x1,y1,x2,y2 = box.tolist()
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                     linewidth=2, edgecolor='red', facecolor='none'))
        ax.text(x1, max(y1-4,0), f'rice {scores_v4[i]:.2f}',
                color='red', fontsize=8, backgroundcolor='black')
    ax.axis('off')
    ax.set_title(f'WeedDet v4 | {len(boxes_v4)} detections '
                 f'(score>={SCORE_THR}) | {img_path.split("/")[-1]}')
    plt.tight_layout(); plt.show()

    print(f'Detections : {len(boxes_v4)}')
    print(f'Score range: {scores_v4.min():.3f}-{scores_v4.max():.3f}'
          if len(scores_v4) else 'No detections above threshold')
    print(f'\nCheckpoints saved at: {CKPT_DIR_V4}')